# Module 02 Assignment: Statistics & Probability

**6 tasks | Auto-graded with assert tests**

Complete each task by replacing `# YOUR CODE HERE` with working code.
Run the assert cell below your solution to verify it.
All tasks use the synthetic WHO health dataset generated in the Setup cell.

> **Do not modify the assert cells.**

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(7)

N = 160
income_groups = ['Low income', 'Lower-middle income', 'Upper-middle income', 'High income']
ig_weights    = [0.25, 0.30, 0.25, 0.20]

income_group = np.random.choice(income_groups, N, p=ig_weights)

le_lookup = {
    'Low income':           (52, 5),
    'Lower-middle income':  (64, 4),
    'Upper-middle income':  (72, 3),
    'High income':          (79, 2),
}

life_expectancy     = np.array([np.clip(np.random.normal(*le_lookup[g]), 35, 90) for g in income_group])
infant_mortality    = np.clip(115 - life_expectancy * 1.7 + np.random.normal(0, 3, N), 2, 120)
health_expenditure  = np.clip((life_expectancy - 50) / 7 + np.random.normal(0, 0.5, N), 1.5, 17)
sanitation_access   = np.clip((life_expectancy - 40) * 1.4 + np.random.normal(0, 5, N), 10, 100)

df = pd.DataFrame({
    'income_group':       income_group,
    'life_expectancy':    life_expectancy.round(1),
    'infant_mortality':   infant_mortality.round(1),
    'health_expenditure': health_expenditure.round(1),
    'sanitation_access':  sanitation_access.round(1),
})

print(f'Dataset ready: {df.shape}')
df.head(3)

---
## Task 1 — Descriptive Statistics (Recall)

Compute the following statistics for the `life_expectancy` column and store
them in variables with the names below:

| Variable | Statistic |
|----------|-----------|
| `le_mean` | Mean (float, rounded to 2 dp) |
| `le_median` | Median (float, rounded to 2 dp) |
| `le_std` | Sample std dev (ddof=1) (float, rounded to 2 dp) |
| `le_iqr` | IQR = Q3 − Q1 (float, rounded to 2 dp) |

Use NumPy or Pandas methods — *not* hard-coded values.

In [ ]:
# YOUR CODE HERE
le_mean   = ...
le_median = ...
le_std    = ...
le_iqr    = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
arr = df['life_expectancy'].values

assert isinstance(le_mean, float), f"le_mean should be float, got {type(le_mean)}"
assert abs(le_mean - round(float(np.mean(arr)), 2)) < 0.01, \
    f"le_mean incorrect: expected {round(float(np.mean(arr)), 2)}, got {le_mean}"

assert abs(le_median - round(float(np.median(arr)), 2)) < 0.01, \
    f"le_median incorrect: expected {round(float(np.median(arr)), 2)}, got {le_median}"

expected_std = round(float(np.std(arr, ddof=1)), 2)
assert abs(le_std - expected_std) < 0.05, \
    f"le_std incorrect: expected {expected_std} (ddof=1), got {le_std}"

expected_iqr = round(float(np.percentile(arr, 75) - np.percentile(arr, 25)), 2)
assert abs(le_iqr - expected_iqr) < 0.1, \
    f"le_iqr incorrect: expected {expected_iqr}, got {le_iqr}"

print('Task 1 passed!')

---
## Task 2 — Bayes' Theorem (Recall)

A disease has a **3% prevalence** in the population being studied.
A diagnostic test has:
- **Sensitivity** (true positive rate) = **88%** — P(positive | disease)
- **Specificity** (true negative rate) = **92%** — P(negative | no disease)

Using Bayes' Theorem, compute `ppv` — the **Positive Predictive Value**:
the probability that someone *actually has the disease* given a positive test.

Round to 4 decimal places.

In [ ]:
# YOUR CODE HERE
prevalence   = 0.03
sensitivity  = 0.88
specificity  = 0.92

ppv = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
p_pos = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
expected_ppv = round((sensitivity * prevalence) / p_pos, 4)

assert isinstance(ppv, float), f"ppv should be float, got {type(ppv)}"
assert 0 < ppv < 1, f"ppv should be a probability between 0 and 1, got {ppv}"
assert abs(ppv - expected_ppv) < 0.001, \
    f"ppv incorrect: expected {expected_ppv}, got {ppv}"
assert ppv < 0.30, \
    "PPV should be well below 30% — the low base rate (3%) drives this down"

print(f'Task 2 passed!  PPV = {ppv:.1%}')
print('  → Even with an 88% sensitive test, only ~{:.0f}% of positive results are true positives.'.format(ppv * 100))

---
## Task 3 — Z-scores and the Normal Distribution (Application)

Using the **High income** group's `life_expectancy` values only:

1. Fit a normal distribution (compute mean and std from the data).
2. Compute `p_above_85` — the probability that a randomly selected
   high-income country has life expectancy **above 85 years**.
   Use `scipy.stats.norm.cdf`. Round to 4 decimal places.
3. Compute `z_score_70` — the Z-score of a life expectancy of 70 years
   relative to the high-income group. Round to 2 decimal places.

In [ ]:
# YOUR CODE HERE
high_income_le = df.loc[df['income_group'] == 'High income', 'life_expectancy'].values

mu_hi = ...
sigma_hi = ...

p_above_85 = ...
z_score_70 = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
hi = df.loc[df['income_group'] == 'High income', 'life_expectancy'].values
exp_mu    = float(np.mean(hi))
exp_sigma = float(np.std(hi, ddof=1))

assert abs(mu_hi - exp_mu) < 0.1, \
    f"mu_hi incorrect: expected {exp_mu:.2f}, got {mu_hi}"
assert abs(sigma_hi - exp_sigma) < 0.1, \
    f"sigma_hi incorrect (use ddof=1): expected {exp_sigma:.2f}, got {sigma_hi}"

exp_p = round(1 - stats.norm.cdf(85, loc=exp_mu, scale=exp_sigma), 4)
assert abs(p_above_85 - exp_p) < 0.005, \
    f"p_above_85 incorrect: expected {exp_p}, got {p_above_85}"
assert 0 <= p_above_85 <= 1, "p_above_85 must be a probability"

exp_z = round((70 - exp_mu) / exp_sigma, 2)
assert abs(z_score_70 - exp_z) < 0.05, \
    f"z_score_70 incorrect: expected {exp_z}, got {z_score_70}"
assert z_score_70 < 0, \
    "70 years is below the high-income group mean — z-score should be negative"

print(f'Task 3 passed!  P(LE > 85 | high income) = {p_above_85:.1%}')
print(f'  Z-score of 70 years: {z_score_70:.2f} std devs below the group mean')

---
## Task 4 — Two-Sample Hypothesis Test (Application)

Test whether **infant mortality** differs significantly between
**Low income** and **High income** countries.

Store the results in:
- `t_stat` — the t-statistic (float, rounded to 2 dp)
- `p_value` — the two-tailed p-value (float, 4 significant figures)
- `reject_h0` — a Python `bool`: `True` if p < 0.05, else `False`

Use Welch's t-test (`equal_var=False`).

In [ ]:
# YOUR CODE HERE
low_im  = df.loc[df['income_group'] == 'Low income',  'infant_mortality'].values
high_im = df.loc[df['income_group'] == 'High income', 'infant_mortality'].values

t_stat    = ...
p_value   = ...
reject_h0 = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
low_im_c  = df.loc[df['income_group'] == 'Low income',  'infant_mortality'].values
high_im_c = df.loc[df['income_group'] == 'High income', 'infant_mortality'].values
exp_t, exp_p = stats.ttest_ind(low_im_c, high_im_c, equal_var=False)

assert isinstance(t_stat, float), f"t_stat should be float, got {type(t_stat)}"
assert abs(t_stat - round(exp_t, 2)) < 0.05, \
    f"t_stat incorrect: expected {round(exp_t, 2)}, got {t_stat}"

assert abs(p_value - exp_p) < 1e-6, \
    f"p_value incorrect: expected {exp_p:.4e}, got {p_value:.4e}"

assert isinstance(reject_h0, bool), f"reject_h0 should be bool, got {type(reject_h0)}"
assert reject_h0 == (exp_p < 0.05), \
    f"reject_h0 should be {exp_p < 0.05}, got {reject_h0}"
assert reject_h0 == True, \
    "Expected to reject H₀ — infant mortality is very different between income groups"

print(f'Task 4 passed!  t = {t_stat:.2f}, p = {p_value:.2e}, reject H₀ = {reject_h0}')

---
## Task 5 — Correlation Analysis (Synthesis)

Compute the Pearson correlation coefficient between `life_expectancy` and
**each of the other three numeric columns** in `df`.

Store the result as a **dictionary** named `correlations` where:
- Keys are the column names: `'infant_mortality'`, `'health_expenditure'`,
  `'sanitation_access'`
- Values are the Pearson r values (float, rounded to 3 dp)

Also store the **name of the column most strongly correlated** (highest |r|)
with life expectancy as a string in `strongest_predictor`.

In [ ]:
# YOUR CODE HERE
target_cols = ['infant_mortality', 'health_expenditure', 'sanitation_access']

correlations = {}
# ... fill the dictionary

strongest_predictor = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
assert isinstance(correlations, dict), \
    f"correlations should be dict, got {type(correlations)}"
assert set(correlations.keys()) == set(target_cols), \
    f"correlations keys don't match: {set(correlations.keys())}"

for col in target_cols:
    exp_r, _ = stats.pearsonr(df['life_expectancy'], df[col])
    exp_r = round(exp_r, 3)
    got_r = correlations[col]
    assert abs(got_r - exp_r) < 0.005, \
        f"r for {col}: expected {exp_r}, got {got_r}"

# Validate strongest predictor
abs_corrs = {k: abs(v) for k, v in correlations.items()}
exp_strongest = max(abs_corrs, key=abs_corrs.get)
assert strongest_predictor == exp_strongest, \
    f"strongest_predictor: expected '{exp_strongest}', got '{strongest_predictor}'"

print('Task 5 passed!')
print('Correlations with life expectancy:')
for k, v in sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {k:30s}: r = {v:+.3f}')
print(f'Strongest predictor: {strongest_predictor}')

---
## Task 6 — Effect Size and Interpretation (Synthesis/Analysis)

Compute **Cohen's d** for the difference in `life_expectancy` between the
**Upper-middle income** and **Lower-middle income** groups.

Cohen's d formula:
```
d = (mean_A - mean_B) / pooled_std
pooled_std = sqrt((std_A² + std_B²) / 2)   # using sample std dev
```

Store the result (float, rounded to 2 dp) in `cohens_d`.

Then store a string interpretation in `effect_size_label`:
- `'small'` if |d| < 0.5
- `'medium'` if 0.5 ≤ |d| < 0.8
- `'large'` if |d| ≥ 0.8

In [ ]:
# YOUR CODE HERE
upper_mid = df.loc[df['income_group'] == 'Upper-middle income', 'life_expectancy'].values
lower_mid = df.loc[df['income_group'] == 'Lower-middle income', 'life_expectancy'].values

cohens_d         = ...
effect_size_label = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
um = df.loc[df['income_group'] == 'Upper-middle income', 'life_expectancy'].values
lm = df.loc[df['income_group'] == 'Lower-middle income', 'life_expectancy'].values

pooled = np.sqrt((np.std(um, ddof=1)**2 + np.std(lm, ddof=1)**2) / 2)
exp_d  = round(float((np.mean(um) - np.mean(lm)) / pooled), 2)

assert isinstance(cohens_d, float), f"cohens_d should be float, got {type(cohens_d)}"
assert abs(cohens_d - exp_d) < 0.05, \
    f"cohens_d incorrect: expected {exp_d}, got {cohens_d}"
assert cohens_d > 0, \
    "Upper-middle income should have higher LE than Lower-middle, so d > 0"

abs_d = abs(cohens_d)
if abs_d < 0.5:
    exp_label = 'small'
elif abs_d < 0.8:
    exp_label = 'medium'
else:
    exp_label = 'large'

assert effect_size_label == exp_label, \
    f"effect_size_label: expected '{exp_label}' for |d|={abs_d:.2f}, got '{effect_size_label}'"

print(f'Task 6 passed!  Cohen\'s d = {cohens_d:.2f} → {effect_size_label} effect')
print(f'  Upper-mid mean: {np.mean(um):.1f} yrs  |  Lower-mid mean: {np.mean(lm):.1f} yrs')
print('\n🎉 All 6 tasks complete! Assignment done.')